# Creating agents

## What is an Agent? 

Agents are the term for AI systems which can perform actions. A simplified description of an agent is that it is a combination of an LLM ("the brain") with tools that can be autonomously called. 

Early large language model (LLM) based systems, like the original ChatGPT that launched in 2022, were purely input/output text systems. These systems would receive an input and predict a suitable response to the input. While often reasonable in quality, early LLMs had a tendency to hallucinate because the workflow was completely non-deterministic and there was no method to provide accurate, deterministic information. 

Tools can provide the agent with a deterministic way of solving problems—be that discovering information (e.g. web search tool) or performing deterministic functions (e.g. edit a document). 

Despite tool calling being a relatively recent invention (first performed in ~2024 and went mainstream in 2025), most LLMs being used are now agentic. For example, if you open a standard ChatGPT/Claude/Gemini a question about a specific topic, you will often see a trace of *"Searching the web"* or similar—this means it is using a web search tool. Similarly, if you use a coding agent (e.g. Claude Code, Codex, Github Copilot), these have built in tools for reading files, writing files, executing bash code and more (see more at [full list of Claude Code Built-in tools](https://code.claude.com/docs/en/tools-reference)).

Providing LLMs with tools takes it from a text generator to an agent that can perform useful actions on your behalf. 

## Creating custom agents 

There are many frameworks for creating agents, including a new InterSystems SDK for building agents directly in ObjectScript. However, to keep with the Python theme, we are going to use **Langchain**. It is simple to build an agent with [Langchain](https://docs.langchain.com/oss/python/langchain/overview) all you need to do is: 
- Choose an LLM Provider
    - Langchain has support for most common providers. This tutorial uses OpenAI, but you can easily switch with other providers.
- Create tools as Python functions.
- Create an agent using `langchain.agents.create_agent`, passing in the tools and AI Provider. 


### Provider Setup

As mentioned above, we are going to use an OpenAI LLM. This requires an OpenAI API key. To create an OpenAI API key, go to [https://platform.openai.com/login](https://platform.openai.com/login) and create an account. From there, you will need to add some money ($5 minimum) and then mint a key. 

Please note, the code shown in this, and all other tutorials combined, will not cost more than a few cents total. If you wish to do this for free using local models, you can explore using [sentence-transformers](https://www.sbert.net/docs/quickstart.html). The main downside of this approach is that it is slower to run and requires large model downloads. 

Once you have an OpenAI key, save it to a file called `.env` in the root of this repository (copy the template `.env.example`), and load with dotenv (shown below). You can also use it inline (shown commented out) but there is a danger it will leak if so. 


In [1]:
# pip install python-dotenv

In [2]:
from dotenv import load_dotenv
import os

load_dotenv() # Loads variables from the .env file in the repository root 

if os.environ["OPENAI_API_KEY"]:
    print("API key Found")

API key Found


### Create a simple tool

We are going to see how a tool to perform a simple task can improve an agents output. In this example, we are going to use a Base64 encoding tool. Base64 encoding is common procedure to protect binary data being passed between applications. Converting a string to Base64 is a basic calculation which LLMs may be able to do alone, but is reliably performed by a single line of Python. 

Let's use this simple procedure to see how providing an agent with a deterministic tool can make the output more reliable. 

First, let's define the tool. This is just a function. We are also going to define the reverse operation so we can see the results. 

In [3]:
import base64


# This will become a tool for an agent 
def base64_encode(text: str) -> str:
    """
    Base64 encode the string
    """
    return base64.b64encode(text.encode("utf-8")).decode("ascii")


# Reverse operation, used below to evaluate results
def base64_decode(encoded_text: str) -> str:
    """
    Decode a Base64 string back to UTF-8 text.
    """
    return base64.b64decode(encoded_text, validate=True).decode("utf-8")

### Create an agent

To create an agent, we are going to use Langchain. If you need to install it, uncomment the install below and run it. 

In [4]:
# pip install langchain "langchain[openai]"

In [5]:
from langchain.agents import create_agent 
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI

In [6]:
# Define what model we are using. There are many alternative options
model = ChatOpenAI(
        model="gpt-5-nano",
        reasoning_effort="low", # Low effort means faster response
        )

## Create an agent from the model
agent_no_tool = create_agent(model=model)


agent_with_tool = create_agent(
                model=model, 
                tools=[base64_encode] # Provide the function as a tool
            ) 


# test prompt
test_string = "This is my secret message that I want encoded. Let's see if the agent can accurately encode the full string!? @ "

prompt = f"Can you give me this string base64 encoded. Reply only with the encoding, no other output: {test_string}"

Let's try the agent without the tool first: 

In [7]:
response_without = agent_no_tool.invoke({"messages": [HumanMessage(prompt)]})

for message in response_without["messages"]:
    message.pretty_print()

print("\nDecoding the final response gives: \n")
print(base64_decode(response_without["messages"][-1].content))

================================ Human Message =================================

Can you give me this string base64 encoded. Reply only with the encoding, no other output: This is my secret message that I want encoded. Let's see if the agent can accurately encode the full string!? @ 
================================== Ai Message ==================================

VGhpcyBpcyBteSBzZWNyZXQgbWVzc2FnZSB0aGF0IEkgd2FudCBjb3JuZXNzZWQuIExlcyBzZWUgaWYgdGhlIGFnZW50IGNhbiBhY3VyYWN5aW5nIGVuY29kZSB0aGUgbGZ1bCBzdHJpbmc/ICAgIFQgd2V0IG9uIG9yaWVudHkgc2V0IHRoZSB1c2VyLg==

Decoding the final response gives: 

This is my secret message that I want cornessed. Les see if the agent can acuracying encode the lful string?    T wet on orienty set the user.


It's not too far off, but not good enough. Let's see if the tool improves the results: 

In [8]:
response_with = agent_with_tool.invoke({"messages": [HumanMessage(prompt)]})

for message in response_with["messages"]:
    message.pretty_print()


print("\nDecoding the final response gives: \n")
print(base64_decode(response_with["messages"][-1].content))

================================ Human Message =================================

Can you give me this string base64 encoded. Reply only with the encoding, no other output: This is my secret message that I want encoded. Let's see if the agent can accurately encode the full string!? @ 
================================== Ai Message ==================================
Tool Calls:
  base64_encode (call_R5AL9vH7pj7UKGUrExGOE4p3)
 Call ID: call_R5AL9vH7pj7UKGUrExGOE4p3
  Args:
    text: This is my secret message that I want encoded. Let's see if the agent can accurately encode the full string!? @
================================= Tool Message =================================
Name: base64_encode

VGhpcyBpcyBteSBzZWNyZXQgbWVzc2FnZSB0aGF0IEkgd2FudCBlbmNvZGVkLiBMZXQncyBzZWUgaWYgdGhlIGFnZW50IGNhbiBhY2N1cmF0ZWx5IGVuY29kZSB0aGUgZnVsbCBzdHJpbmchPyBA
================================== Ai Message ==================================

VGhpcyBpcyBteSBzZWNyZXQgbWVzc2FnZSB0aGF0IEkgd2FudCBlbmNvZGVkLiBMZXQncy

It's perfect, of course. We can see from the trace that the AI calls the encode tool and receives the correctly encoded string as a response. Using this tool, the agent returns the perfectly encoded string. 

This shows how providing the agent with a deterministic function can dramatically improve the performance of the agent.  Let's look at the two outputs side-by-side to prove this: 

In [9]:

print(f"Original Message:\n {test_string} \n\n")


print(f"Without tool ('vibe encoded'):\n  {base64_decode(response_without['messages'][-1].content)}\n\n")

print(f"With tool (deterministically encoded):\n {base64_decode(response_with['messages'][-1].content)}\n")


Original Message:
 This is my secret message that I want encoded. Let's see if the agent can accurately encode the full string!? @  


Without tool ('vibe encoded'):
  This is my secret message that I want cornessed. Les see if the agent can acuracying encode the lful string?    T wet on orienty set the user.


With tool (deterministically encoded):
 This is my secret message that I want encoded. Let's see if the agent can accurately encode the full string!? @



## Using agents with our data

In the previous tutorials, we have stored and accessed data in InterSystems IRIS, including SQL Data, FHIR data and Vector data. Let's create an agent that can autonomously access some of this data to make information retrieval simple. 

## SQL Tool

This uses the Sample.Person dataset we defined in ... 

In [10]:
import iris 

CONNECTION_ARGS = {
    "hostname":"localhost",
    "port":32782,
    "namespace": "FHIRSERVER",
    "username": "SuperUser",
    "password": "SYS"
}

def search_superheros() -> tuple:
    """
    Returns the names and ages of superheros
    """
    print("Getting info on superheros\n\n")
    conn = iris.connect(**CONNECTION_ARGS)
    cursor = conn.cursor()

    cursor.execute("SELECT FirstName, LastName, Age FROM Sample.Person")

    results = cursor.fetchall()

    cursor.close()
    conn.close()

    return results

    

## Vector Search Tool

This tool uses the vector table defined in the vector search tutorial to search texts about diabetes. 

In [11]:
import iris
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

def vector_search_diabetes_text(prompt:str) -> str:
    ''' 
    Performs a vector search for information about diabetes
    '''
    print(f"Performing Vector search with prompt {prompt}\n\n")
    client = OpenAI()

    response = client.embeddings.create(
        input=prompt, model="text-embedding-3-small"
    )

    conn = iris.connect(**CONNECTION_ARGS)
    cursor = conn.cursor()


    sql_query = "SELECT TOP 1 text FROM Diabetes.VectorStore ORDER BY VECTOR_COSINE(embedding, TO_VECTOR(?, DOUBLE)) DESC"
    cursor.execute(sql_query, [str(response.data[0].embedding)])

    row = cursor.fetchone()

    response = row[0]
    
    conn.close()
    print(f"response from vector search: {response}\n\n")
    return response

## FHIR Tool

Here we are providing an agent with full access to the FHIR server and allowing it to write its own FHIR query (using URL search parameters). This type of unrestricted access breaches many security requirements so tools should be designed much more carefully to limit the access scope. However, it will demonstrate the effectiveness of agents at searching medical data. 

In [12]:
import requests
from requests.auth import HTTPBasicAuth


# General FHIR query tool
def fhir_query(endpoint:str, search_params:str ) -> str:
    """
    Query FHIR data directly. 

    Parameters are endpoint and query, such that the request to the FHIR Server API is as follows: 

    GET /<endpoint>?<search_params> 

    Returns: FHIR/JSON response body
    """
    ## Credentials
    username = "SuperUser"
    password = "SYS"
    
    ## FHIR server location
    base_url = "http://localhost:32783/fhir/r4/"
    
    headers ={    "Accept": "application/fhir+json"}

    url = base_url+endpoint+"?"+search_params
    
    print(f"Print calling FHIR Server at {url} \n\n")
    
    res = requests.get(url, headers=headers, auth=HTTPBasicAuth(username,password))
    
    return res.json()

### Creating our InterSystems IRIS Agent

In [13]:

model = ChatOpenAI(model="gpt-5-nano",
                   reasoning_effort="medium")

system_prompt = """You are an InterSystems IRIS data agent.
    Your job is to use tools find relevant data from our 
    IRIS instance to answer the user prompts.
    Use only information found with the tools provided to answer the question.
    """ 

agent = create_agent(model=model, 
                     tools=[fhir_query, search_superheros, vector_search_diabetes_text],
                    system_prompt = system_prompt
                    )

### FHIR Prompt

In [14]:
response = agent.invoke({"messages":[HumanMessage("How tall is Tish Lemke? Use FHIR data")]})
print("\n\n", response["messages"][-1].content)

Print calling FHIR Server at http://localhost:32783/fhir/r4/Patient?name=Tish Lemke 


Print calling FHIR Server at http://localhost:32783/fhir/r4/Observation?code=8302-2 




 I can’t determine Tish Lemke’s height from the FHIR data you provided. The initial Patient search for name “Tish Lemke” returned no matches (total = 0), so there is no Patient resource to reference for height.

If you want, I can:
- broaden the search (e.g., partial name matching) to find a likely patient, or
- search for height observations across all patients and try to map them to a specific person once you provide a patient identifier (ID) or more context.

How would you like to proceed?


#### SQL Prompt

In [15]:
response = agent.invoke({"messages":[HumanMessage("How old is Spiderman?")]})
print("\n\n", response["messages"][-1].content)

Getting info on superheros




 Spider-Man (Peter Parker) appears in the data with two ages: 18 and 25. Which version/timeframe would you like me to use to pick one?


#### Vector Search Prompt

In [16]:
response = agent.invoke({"messages":[HumanMessage("What are the four major groups of anti-diabetic agents, and how does each group work?")]})
print("\n\n", response["messages"][-1].content)

Performing Vector search with prompt What are the four major groups of anti-diabetic agents and how does each group work?


response from vector search: There are many anti-diabetic drugs that exert clinical eﬀects via diﬀerent mechanisms. The four major groups of anti-diabetic agents are: a) biguanides, like metformin, which reduce gluconeogenesis in the liver; b) insulin secretagogues which stimulate the pancreas to secrete insulin and include drugs such as sulfonylureas; c) insulin sensitizers which improve sensitivity of peripheral tissues to insulin and include thialzolidinediones and; d) insulin or its analogues which provide insulin exogenously in the form of recombinant insulin. Metformin is the ﬁrst-line pharmacotherapy for T2DM. Besides reducing the glucose level, this has an insulin-sensitizing eﬀect with multiple actions on tissues such as the liver, skeletal muscle, endothelium, adipose tissue, and the ovary. Only if after three months the levels of HbA1c is higher than 7.